In [1]:
import os
import pickle
import scanpy as sc
import numpy as np
import process_model
import scipy.stats as stats

import warnings
warnings.filterwarnings("ignore")

import importlib
importlib.reload(process_model)

<module 'process_model' from '/home/masse/work/AMPPD_Trajectory/src/process_model.py'>

## Coverting PyTorch Lightning model outputs to an h5ad
This required for all downstream analysis.  
process_data will add donor-related information (e.g. average predicted Braak, average gene expression) to the h5ad.
This worksheet shows an example of how to perform this conversion.

In [2]:
# Helper function to measure model accuracy

def explained_var(x_pred, x_real):
    
    idx = ~np.isnan(x_real) * (x_real > -1)
    x = x_real[idx]
    y = x_pred[idx]
    ex_var = 1 - np.nanvar(x - y) / np.nanvar(x)
    r, p = stats.pearsonr(x, y)
    return r, ex_var

In [6]:
# When save data is True, model results will be saved as an h5ad after running create_data below
# Can be set to False to first measure model accuracy across different epochs
save_data = False

In [7]:
p = "path_braak_lb_condensed"

mr = process_model.ModelResults(
    data_fn= "../data/PD_myeloid/data.dat",
    meta_fn = "../data/PD_myeloid/metadata.pkl",
    obs_list= [p],
    normalize_gene_counts=True,
    log_gene_counts=True,
    add_all_gene_vals=False,
    add_gene_scores=save_data,
)

In [8]:

# where PyTorch Lightning model outputs are saved
base_path = "logs/AMPPD_PD/"

# where the processed model output (.h5ad) will be saved
save_dir = "../model_output/processed_model_files/"

# min_cell_count only used when measuring accuracy, does not affect saved model
min_cell_count = 20

# the 20 splits be saved in different directories
# version_nums indicate which versions to look at
version_nums = np.arange(0, 20)

# 40 - 60 is currently the best

# can average the results over ultiple epochs if desired
n_epochs_to_avg = 5

model_save_fn = "PD_myeloid.h5ad"

# we loop through epochs and monitor how accuracy changes, but we only save the results of the last specified epoch
# feel free to chage range
for epoch in range(0, 10 - n_epochs_to_avg):  
    
    fns = []
    for n in range(n_epochs_to_avg):
        fns_temp = []
        for v in version_nums:
            fns_temp.append(os.path.join(base_path, f"version_{v}/test_results_ep{epoch + n}.pkl"))
        fns.append(fns_temp)

    adata = mr.create_data(fns, model_average=True, subclass=None, brain_region=None)

    # single-cell results
    pred = adata.obs[f"pred_{p}"].values
    r, ev = explained_var(pred, adata.obs["path_braak_lb"].values)

    # donor results
    idx = np.where(adata.uns["donor_cell_count"] >= min_cell_count)[0]
    r_donor, ev_donor = explained_var(adata.uns[f"donor_pred_{p}"][idx], adata.uns[f"donor_path_braak_lb"][idx])
    
    print(f"{epoch}, R_cell={r:1.3f}, R_donor={r_donor:1.3f}, ExVar={ev:1.3f}, ExVar_donor={ev_donor:1.3f}")


if save_data:
    fn = os.path.join(save_dir, model_save_fn)
    adata.write(fn)
    print(f"Model saved to: {fn}")

Subclasses present: ['Mg_Adapt' 'PVM' 'Mg_Homeo' 'Mg_DAM' 'Mg_Prolif' 'CD11a+']
0, R_cell=0.348, R_donor=0.388, ExVar=0.077, ExVar_donor=0.149
Subclasses present: ['Mg_Adapt' 'PVM' 'Mg_Homeo' 'Mg_DAM' 'Mg_Prolif' 'CD11a+']
1, R_cell=0.362, R_donor=0.393, ExVar=0.043, ExVar_donor=0.140
Subclasses present: ['Mg_Adapt' 'PVM' 'Mg_Homeo' 'Mg_DAM' 'Mg_Prolif' 'CD11a+']
2, R_cell=0.371, R_donor=0.409, ExVar=0.038, ExVar_donor=0.149
Subclasses present: ['Mg_Adapt' 'PVM' 'Mg_Homeo' 'Mg_DAM' 'Mg_Prolif' 'CD11a+']
3, R_cell=0.378, R_donor=0.426, ExVar=0.081, ExVar_donor=0.176
Subclasses present: ['Mg_Adapt' 'PVM' 'Mg_Homeo' 'Mg_DAM' 'Mg_Prolif' 'CD11a+']
4, R_cell=0.384, R_donor=0.429, ExVar=0.130, ExVar_donor=0.184
